# 02 - Bronze to Silver Transformation

**Project:** Gender-Based Sales KPI Dashboard  
**Author:** Hicham ERRIHANI  
**Layer:** Bronze → Silver  

This notebook cleans, types, and validates raw data from the Bronze layer, then writes it to the Silver layer.

In [ ]:
# Imports
from pyspark.sql.functions import col, when, to_date, trim, upper
from pyspark.sql.types import IntegerType, DoubleType, StringType

# Configuration
STORAGE_ACCOUNT = "stgendersaleshicham"
BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

In [ ]:
# Read Bronze tables
dim_customer = spark.read.parquet(f"{BRONZE_PATH}DimCustomer/")
fact_sales = spark.read.parquet(f"{BRONZE_PATH}FactInternetSales/")

print(f"DimCustomer rows: {dim_customer.count()}")
print(f"FactInternetSales rows: {fact_sales.count()}")

In [ ]:
# Clean DimCustomer
dim_customer_clean = dim_customer \
    .withColumn("Gender", upper(trim(col("Gender")))) \
    .withColumn("YearlyIncome", col("YearlyIncome").cast(DoubleType())) \
    .withColumn("TotalChildren", col("TotalChildren").cast(IntegerType())) \
    .dropDuplicates(["CustomerKey"]) \
    .filter(col("CustomerKey").isNotNull())

print(f"DimCustomer (cleaned) rows: {dim_customer_clean.count()}")

In [ ]:
# Clean FactInternetSales
fact_sales_clean = fact_sales \
    .withColumn("SalesAmount", col("SalesAmount").cast(DoubleType())) \
    .withColumn("OrderDate", to_date(col("OrderDate"), "yyyy-MM-dd")) \
    .filter(col("SalesAmount") > 0) \
    .dropDuplicates(["SalesOrderNumber", "SalesOrderLineNumber"])

print(f"FactInternetSales (cleaned) rows: {fact_sales_clean.count()}")

In [ ]:
# Write to Silver layer (Parquet)
dim_customer_clean.write.mode("overwrite").parquet(f"{SILVER_PATH}DimCustomer/")
fact_sales_clean.write.mode("overwrite").parquet(f"{SILVER_PATH}FactInternetSales/")

print("Silver layer written successfully.")